In [1]:
import amulet
import os
import sys
import json
import numpy as np
import pandas as pd
from amulet import load_level
from amulet_nbt import load
from amulet.api.selection import SelectionBox
from collections import namedtuple
import re
import mcschematic
from gdpc import Block
from gdpc.block import transformedBlockOrPalette
import itertools

INFO - PyMCTranslate Version 385


In [2]:
data_path = '../../data/'
palette = 'java_palette_copy.json'
palette_path = os.path.join(data_path, palette)

In [3]:
class Palette:
    
    def __init__(self, src: str | dict | list):
        # Get the block to token mapping
        if isinstance(src, str):
            with open(src, 'r') as file:
                self.block2token = json.load(file)
        elif isinstance(src, dict):
            self.block2token = src
        elif isinstance(src, list):
            self.block2token = {k : i for i, k in enumerate(src)}
        else:
            raise ValueError("src argument should be a file path (str), block2token mapping (dict), or list of blocks (list)")

        # Validate that all block strings are valid
        self._blockstr_pattern = re.compile(r"^minecraft:[a-z0-9_]+(?:\[[a-z0-9_]+=[a-z0-9_]+(?:,[a-z0-9_]+=[a-z0-9_]+)*\])?$")
        self._blockstr_separation_pattern = pattern = re.compile(r"(minecraft:[^\[]+)(?:\[(.*)\])?")
        
        invalid_blocks = []
        for blockstr in self.block2token.keys():
            if not self._valid_blockstr(blockstr):
                invalid_blocks.append(blockstr)
        
        if invalid_blocks:
            raise ValueError(f"The following block strings are invalid: {invalid_blocks}")
            
        # Get other mapping lists
        self.block_strings = [blockstr for blockstr in self.block2token.keys()]
        self.token2block = {v:k for k,v in self.block2token.items()}
        self.gdpc_blocks = [self._blockstr_to_gdpc_block(blockstr) for blockstr in self.block_strings]
        self.length = len(self.block_strings)
        
        self._add_missing_transformations()
        
        self.TransformLookup = namedtuple('TransformLookup', ['rot0', 'rot0flip', 'rot90', 'rot90flip', 'rot180', 'rot180flip', 'rot270', 'rot270flip'])
        lookups = self._generate_transformation_lookups()
        self.transform_lookup = self.TransformLookup(*lookups)
        
        
    def reduce_blockstates(self, keep_blockstates: list) -> tuple["Palette", np.ndarray, np.ndarray]:
        reduced_block2tok = {}
        reduced_gdpc_blocks = []
        src2tgt_lookup = np.zeros(self.length, dtype=np.int16)
        
        for blockstr, token in self.block2token.items():
            block_id, states = self._blockstr_to_id_states(blockstr)
            
            # remove any blockstate info were not keeping
            if states: states = {k:v for k,v in states.items() if k in keep_blockstates}
            
            # Using GDPC block objects since they can tell if two blocks with different state orders are equal
            reduced_block = Block(block_id, states)
            
            # add to the new palette if its not already in
            if reduced_block not in reduced_gdpc_blocks:
                reduced_gdpc_blocks.append(reduced_block)
                reduced_block2tok[str(reduced_block)] = len(reduced_block2tok)
            
            # Now, map this block to the reduced palette
            reduced_token = reduced_block2tok[str(reduced_block)]
            src2tgt_lookup[token] = reduced_token
        
        # Now, we reverse the lookup table so we can convert back to the original palette. The tgt tokens will map to the first instance in the src that maps to it
        tgt2src_lookup = np.full(len(reduced_block2tok), -1, dtype=np.int16)
        
        for i, token in enumerate(src2tgt_lookup):
            if tgt2src_lookup[token] == -1:
                tgt2src_lookup[token] = i
        
        # Now, construct the new Palette
        reduced_palette = Palette(reduced_block2tok)
        
        return reduced_palette, src2tgt_lookup, tgt2src_lookup
    
    def _generate_transformation_lookups(self):
        rotations = [0,1,2,3]
        flips = [0,1]
        combos = list(itertools.product(rotations, flips))
        lookup_arrays = []
        
        for rotation, flip in combos:
            transformed_blocks = transformedBlockOrPalette(block=self.gdpc_blocks, rotation=rotation, flip=(flip,0,0))
            lookup_arr = np.full(self.length, -1, dtype=np.int16)
            for i, block in enumerate(transformed_blocks):
                lookup_arr[i] = self.block2token[str(block)]
            
            assert -1 not in lookup_arr
            lookup_arrays.append(lookup_arr)
        
        return lookup_arrays
            
    def _add_missing_transformations(self):
        count_added = 0
        rotations = [0,1,2,3]
        flips = [0,1]
        combos = list(itertools.product(rotations, flips))
        
        for rotation, flip in combos:
            all_possible_blocks = transformedBlockOrPalette(block=self.gdpc_blocks, rotation=rotation, flip=(flip,0,0))
            for block in all_possible_blocks:
                if block not in self.gdpc_blocks:
                    blockstr = str(block)
                    self.gdpc_blocks.append(block)
                    self.block_strings.append(blockstr)
                    self.block2token[blockstr] = len(self.block2token)
                    self.token2block[self.block2token[blockstr]] = blockstr
                    count_added += 1
        self.length = len(self.block_strings)
        print(f'Added {count_added} new blocks')
                
    
    def _valid_blockstr(self, blockstr: str) -> bool:
        return bool(self._blockstr_pattern.fullmatch(blockstr))
    
    def _blockstr_to_id_states(self, blockstr: str):
        block_id, states_str = self._blockstr_separation_pattern.fullmatch(blockstr).groups()
        states = dict(p.split("=") for p in states_str.split(",")) if states_str else None
        return block_id, states
    
    def _blockstr_to_gdpc_block(self, blockstr):
        block_id, states = self._blockstr_to_id_states(blockstr)
        return Block(block_id, states)
    
    def __str__(self):
        lines = [f"{token}: {block}" for block, token in self.block2token.items()]
        return '\n'.join(lines)
    

In [4]:
java_palette = Palette(palette_path)

keep_blockstates = ["axis", "facing", "shape"]
keep_blockstates = ["axis", "facing", "shape", "east", "west", "north", "south", "face", "half"]


Added 556 new blocks


In [5]:
reduced_palette, src2tgt, tgt2src = java_palette.reduce_blockstates(keep_blockstates)

Added 0 new blocks


In [6]:
print(len(src2tgt))
print(len(tgt2src))


9498
5812


In [7]:
def array_to_schematic(token_array, tok2block, save_path, filename="my_schematic"):
    """
    Converts a 3D numpy array of integer tokens into a Minecraft schematic file.

    Args:
        token_array (np.ndarray): A 3D array where values correspond to block types.
        filename (str): The name of the output schematic file (without extension).
    """
    schem = mcschematic.MCSchematic()

    d, h, w = token_array.shape

    # Iterate through the 3D array and place blocks
    for x in range(d):
        for y in range(h):
            for z in range(w):
                token = token_array[x, y, z]
                # block_name = tok2block.get(f'{token}', "minecraft:air") # Default to air if token not found
                block_name = tok2block[token]
                

                # Place the block in the schematic at the specified coordinates (x, y, z)
                schem.setBlock((x, y, z), block_name)

    # Save the schematic file
    schem.save(save_path, filename, mcschematic.Version.JE_1_21_5, True)
    print(f"Successfully saved schematic to {filename}.schem")

In [8]:
schem_name = 'build_batch_65_1671_1'
schem_name = 'build_batch_23_575_1'
schem_path = os.path.join(data_path, 'new_data/java/', f'{schem_name}_java.npy')
sample = np.load(schem_path)

In [9]:
reduced_sample = src2tgt[sample]
reverted_sample = tgt2src[reduced_sample]

In [10]:
array_to_schematic(reverted_sample, java_palette.token2block, '../', 'test_revert')

Successfully saved schematic to test_revert.schem


In [11]:
air_token = reduced_palette.block2token["minecraft:air"]
north_inner_left = reduced_palette.block2token["minecraft:oak_stairs[facing=north,half=bottom,shape=inner_left]"]
north_inner_right = reduced_palette.block2token["minecraft:oak_stairs[facing=north,half=bottom,shape=inner_right]"]
south_inner_left = reduced_palette.block2token["minecraft:oak_stairs[facing=south,half=bottom,shape=inner_left]"]
south_inner_right = reduced_palette.block2token["minecraft:oak_stairs[facing=south,half=bottom,shape=inner_right]"]



sample_arr = np.zeros((2,2,2), dtype=np.int16)
sample_arr[:,1,:] = air_token
sample_arr[0,0,0] = north_inner_left
sample_arr[1,0,0] = north_inner_right
sample_arr[0,0,1] = south_inner_right
sample_arr[1,0,1] = south_inner_left

sample_to_game = tgt2src[sample_arr]
array_to_schematic(sample_to_game, java_palette.token2block, '', 'orig_sample')

flip = (1,0,0)
rotation = 3
test_rotation_sample = np.copy(sample_arr)

for axis, i in enumerate(flip):
    if i: test_rotation_sample = np.flip(test_rotation_sample, axis=axis)
test_rotation_sample = np.rot90(test_rotation_sample, rotation, (0,2))

test_rotation_sample = reduced_palette.transform_lookup.rot270flip[test_rotation_sample]
array_to_schematic(tgt2src[test_rotation_sample], java_palette.token2block, '', 'transform_sample')

Successfully saved schematic to orig_sample.schem
Successfully saved schematic to transform_sample.schem


In [12]:
test_rotation_sample = np.copy(sample)
test_rotation_sample = np.flip(test_rotation_sample, axis=0)
test_rotation_sample = np.rot90(test_rotation_sample, 1, (0,2))
test_rotation_sample = java_palette.transform_lookup.rot90flip[test_rotation_sample]
array_to_schematic(test_rotation_sample, java_palette.token2block, '../', 'test_rotation')

Successfully saved schematic to test_rotation.schem


In [13]:
block_str = "minecraft:oxidized_cut_copper_stairs[facing=north,half=bottom,shape=inner_left]"

pattern = re.compile(r"(minecraft:[^\[]+)(?:\[(.*)\])?")
block_id, states_str = pattern.fullmatch(block_str).groups()
states = dict(p.split("=") for p in states_str.split(",")) if states_str else None

keep_blockstates = ['axis', 'facing', 'shape']

In [14]:
block_obj = Block(block_id, states, None)
print(block_obj.id)
print(block_obj.states)

minecraft:oxidized_cut_copper_stairs
{'facing': 'north', 'half': 'bottom', 'shape': 'inner_left'}
